# PhysDiff — physics-guided diffusion, minimal example

**Paper:** Yuan, Song, Iqbal, Vahdat, Kautz, *PhysDiff: Physics-Guided Human Motion Diffusion Model*, ICCV 2023 —
https://arxiv.org/abs/2212.02500 · project page: https://nvlabs.github.io/PhysDiff/

**Core idea.** A standard motion diffusion model (MDM-style transformer that predicts the clean motion `x̂0`)
often produces physically implausible motion: feet floating or sinking into the ground, foot skating, jitter.
PhysDiff inserts a **physics projection** `P(·)` into the *sampling loop*: at selected diffusion steps,
the predicted clean motion is projected onto the set of physically plausible motions, and the next noisy sample
`x_{t-1}` is drawn from the posterior `q(x_{t-1} | x_t, P(x̂0))`. Training is unchanged.

In the paper `P` is a motion-imitation RL policy tracking `x̂0` in a physics simulator (IsaacGym).
**Here `P` is a lightweight, differentiable-free stand-in** that enforces three constraints directly:
ground penetration, foot skating, and bone-length consistency. That keeps the example runnable anywhere while
showing exactly where the projection sits in the sampler. Swap `physics_projection` for a simulator to get the
real method. The same pattern (diffusion + constraint projection) is useful for physical fields as well,
e.g. enforcing non-negativity or mass conservation on generated weather/climate states.

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

## A toy skeleton

7 joints, y-axis up. A motion is a tensor `(B, T, J, 3)` of joint positions.

In [2]:
JOINTS = ["pelvis", "l_knee", "l_foot", "r_knee", "r_foot", "spine", "head"]
PARENTS = [-1, 0, 1, 0, 3, 0, 5]
FEET = [JOINTS.index("l_foot"), JOINTS.index("r_foot")]
REST = torch.tensor([[0.0, 0.9, 0.0], [0.1, 0.5, 0.0], [0.1, 0.05, 0.0], [-0.1, 0.5, 0.0],
                     [-0.1, 0.05, 0.0], [0.0, 1.3, 0.0], [0.0, 1.65, 0.0]])
BONE_LEN = torch.stack([(REST[j] - REST[p]).norm() if p >= 0 else torch.tensor(0.0)
                        for j, p in enumerate(PARENTS)])
J, T = len(JOINTS), 60


def synthetic_walks(n, period=30):
    """Cheap, physically clean training data: a planted stance foot and a lifted swing foot."""
    t = torch.arange(T, dtype=torch.float32)
    stride = 0.25 + 0.15 * torch.rand(n, 1)  # metres per half gait cycle
    x = REST.expand(n, T, J, 3).clone()
    for j in (0, 5, 6):  # upper body moves forward at constant speed
        x[:, :, j, 2] += stride * t / (period / 2)
    for knee, foot, offset in ((1, 2, 0.0), (3, 4, 0.5)):
        phase = t / period + offset
        k, frac = phase.floor(), phase - phase.floor()
        s = ((frac - 0.5) / 0.5).clamp(0, 1)
        s = s * s * (3 - 2 * s)  # smoothstep: zero velocity at lift-off and touch-down
        x[:, :, foot, 2] = stride * 2 * (k + s - offset) + stride
        x[:, :, foot, 1] += 0.08 * torch.sin(math.pi * s)  # lift only during swing
        x[:, :, knee, 2] = 0.5 * (x[:, :, 0, 2] + x[:, :, foot, 2])
    return physics_projection(x)  # snap knees to exact bone lengths (defined below)

## Denoiser: MDM-style transformer predicting `x̂0`

In [3]:
class TimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.float()[:, None] * freqs[None]
        return self.mlp(torch.cat([args.sin(), args.cos()], dim=-1))


class MotionDenoiser(nn.Module):
    def __init__(self, n_feats=J * 3, dim=256, depth=4, heads=4, max_len=256):
        super().__init__()
        self.inp = nn.Linear(n_feats, dim)
        self.pos = nn.Parameter(torch.randn(1, max_len + 1, dim) * 0.02)
        self.t_emb = TimestepEmbedding(dim)
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 4, dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth)
        self.out = nn.Linear(dim, n_feats)

    def forward(self, x_t, t):  # x_t: (B, T, J, 3) -> x̂0: (B, T, J, 3)
        B, T_, _, _ = x_t.shape
        h = self.inp(x_t.reshape(B, T_, -1))
        h = torch.cat([self.t_emb(t)[:, None], h], dim=1) + self.pos[:, : T_ + 1]  # timestep as a prefix token
        h = self.encoder(h)[:, 1:]
        return self.out(h).reshape(B, T_, J, 3)

## Gaussian diffusion (DDPM, cosine schedule, `x0` parameterisation)

In [4]:
class Diffusion:
    def __init__(self, steps=100):
        self.steps = steps
        s = 0.008
        f = torch.cos((torch.linspace(0, steps, steps + 1) / steps + s) / (1 + s) * math.pi / 2) ** 2
        abar = f / f[0]
        self.betas = (1 - abar[1:] / abar[:-1]).clamp(max=0.999)
        self.alphas = 1 - self.betas
        self.abar = torch.cumprod(self.alphas, 0)
        self.abar_prev = torch.cat([torch.ones(1), self.abar[:-1]])
        # posterior q(x_{t-1} | x_t, x0) coefficients
        self.post_var = self.betas * (1 - self.abar_prev) / (1 - self.abar)
        self.c0 = self.betas * self.abar_prev.sqrt() / (1 - self.abar)
        self.ct = (1 - self.abar_prev) * self.alphas.sqrt() / (1 - self.abar)

    def q_sample(self, x0, t, noise):
        a = self.abar.to(x0.device)[t].view(-1, 1, 1, 1)
        return a.sqrt() * x0 + (1 - a).sqrt() * noise

    def posterior_sample(self, x0, x_t, t):
        c0, ct = self.c0[t].item(), self.ct[t].item()
        mean = c0 * x0 + ct * x_t
        if t == 0:
            return mean
        return mean + self.post_var[t].sqrt().item() * torch.randn_like(x_t)

## Physics projection `P(x̂0)`

Stand-in for the simulator-based imitation step. Each operation pushes the motion toward the plausible set:
1. **Bone lengths** — re-project every child joint to the correct distance from its parent.
2. **Ground penetration** — lift the whole body so the lowest joint is not below the floor (y = 0).
3. **Foot skating** — when a foot is in contact (near the floor), pin its horizontal position to the previous frame.

In [5]:
def physics_projection(x, contact_h=0.08, iters=5):
    """The constraints interact (pinning a foot changes a bone length), so alternate between them a few times."""
    x = x.clone()
    bl = BONE_LEN.to(x.device)
    for _ in range(iters):
        # 1) foot skating: pin feet that stay in contact
        for f in FEET:
            for t in range(1, x.shape[1]):
                contact = (x[:, t, f, 1] < contact_h) & (x[:, t - 1, f, 1] < contact_h)
                x[:, t, f, 0] = torch.where(contact, x[:, t - 1, f, 0], x[:, t, f, 0])
                x[:, t, f, 2] = torch.where(contact, x[:, t - 1, f, 2], x[:, t, f, 2])
        # 2) bone lengths: move each knee so both thigh and shin have the right length
        #    (intersection of two spheres; the knee bends forward)
        for knee, foot in ((1, 2), (3, 4)):
            hip, ft = x[:, :, 0], x[:, :, foot]
            d = ft - hip
            dist = d.norm(dim=-1, keepdim=True).clamp(min=1e-6)
            a, b = bl[knee], bl[foot]
            dist_c = dist.clamp(max=a + b - 1e-4)
            x[:, :, foot] = hip + d / dist * dist_c  # a straight leg is the furthest the foot can reach
            along = (a ** 2 - b ** 2 + dist_c ** 2) / (2 * dist_c)
            h = (a ** 2 - along ** 2).clamp(min=0).sqrt()
            u = d / dist
            fwd = torch.tensor([0.0, 0.0, 1.0], device=x.device).expand_as(u)
            perp = fwd - (fwd * u).sum(-1, keepdim=True) * u
            perp = perp / perp.norm(dim=-1, keepdim=True).clamp(min=1e-6)
            x[:, :, knee] = hip + u * along + perp * h
        for j in (5, 6):  # spine and head: rigid chain
            p = PARENTS[j]
            d = x[:, :, j] - x[:, :, p]
            x[:, :, j] = x[:, :, p] + d / d.norm(dim=-1, keepdim=True).clamp(min=1e-6) * bl[j]
        # 3) ground penetration: lift the body so the lowest joint is on or above the floor
        lowest = x[..., 1].amin(dim=2, keepdim=True)
        x[..., 1] = x[..., 1] - lowest.clamp(max=0)
    return x


def physics_metrics(x, contact_h=0.08):
    penetration = (-x[..., 1]).clamp(min=0).mean().item()
    feet = x[:, :, FEET]
    in_contact = (feet[:, 1:, :, 1] < contact_h) & (feet[:, :-1, :, 1] < contact_h)
    slide = (feet[:, 1:, :, [0, 2]] - feet[:, :-1, :, [0, 2]]).norm(dim=-1)
    skating = (slide * in_contact).sum().item() / max(in_contact.sum().item(), 1)
    bone_err = torch.stack([((x[:, :, j] - x[:, :, p]).norm(dim=-1) - BONE_LEN[j]).abs().mean()
                            for j, p in enumerate(PARENTS) if p >= 0]).mean().item()
    return {"penetration": round(penetration, 4), "skating": round(skating, 4), "bone_err": round(bone_err, 4)}

## Training (unchanged from a normal motion diffusion model)

A short run on the synthetic walks, just enough for the sampler to produce motion-shaped output.

In [6]:
diff = Diffusion(steps=100)
model = MotionDenoiser().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

for it in range(300):
    x0 = synthetic_walks(64).to(device)
    t = torch.randint(0, diff.steps, (x0.shape[0],), device=device)
    noise = torch.randn_like(x0)
    loss = F.mse_loss(model(diff.q_sample(x0, t, noise), t), x0)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if it % 100 == 0 or it == 299:
        print(f"iter {it:4d}  loss {loss.item():.4f}")

/tmp/ipykernel_1122777/3736056856.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, depth)


params: 3.4M


iter    0  loss 1.6356


iter  100  loss 0.0413


iter  200  loss 0.0238


iter  299  loss 0.0123


## Sampling with and without physics guidance

The paper finds it works best to apply the projection only at a few late steps (it is expensive with a real
simulator, and early `x̂0` predictions are too blurry to be worth projecting). `physics_steps` controls that.

In [7]:
@torch.no_grad()
def sample(model, diff, n, physics_steps=()):
    model.eval()
    x = torch.randn(n, T, J, 3, device=device)
    for t in reversed(range(diff.steps)):
        x0_hat = model(x, torch.full((n,), t, device=device, dtype=torch.long))
        if t in physics_steps:
            x0_hat = physics_projection(x0_hat)  # PhysDiff: project, then continue the reverse process
        x = diff.posterior_sample(x0_hat, x, t)
    return x


torch.manual_seed(1)
plain = sample(model, diff, 16)
torch.manual_seed(1)
physd = sample(model, diff, 16, physics_steps=set(range(0, 20, 4)))

print("plain diffusion:", physics_metrics(plain.cpu()))
print("PhysDiff-style: ", physics_metrics(physd.cpu()))
print("training data:  ", physics_metrics(synthetic_walks(16)))
# The projection should fix violations without rewriting the motion: mean joint displacement vs plain sampling (m)
print("mean joint shift:", round((physd - plain).norm(dim=-1).mean().item(), 4))

plain diffusion: {'penetration': 0.0, 'skating': 0.0208, 'bone_err': 0.024}
PhysDiff-style:  {'penetration': 0.0, 'skating': 0.0, 'bone_err': 0.0}
training data:   {'penetration': 0.0, 'skating': 0.0, 'bone_err': 0.0}
mean joint shift: 0.0342
